# GBasis libcint Tutorial

This notebook demonstrates how to use the `CBasis` class from `gbasis.integrals.libcint` to compute GTO integrals using C shell-loop implementations backed by the [libcint](https://github.com/sunqm/libcint) library.

**What you will learn:**
- How to set up a `CBasis` instance for a molecule
- How to compute 1-electron integrals (overlap, kinetic, nuclear attraction)
- How to compute 2-electron repulsion integrals (ERI)
- How to compute gradient integrals (building blocks for nuclear gradients)
- How to compute GIAO/magnetic integrals (building blocks for NMR properties)
- How to compute 3-center 2-electron integrals (used in density fitting)

**Prerequisites:**
- GBasis installed with libcint support (`pip install gbasis`)
- A basis set file in NWChem format (e.g. `sto-3g.nwchem`)


## Installation

### Prerequisites
- Python 3.9+
- C compiler (GCC on Linux/Windows, Clang on macOS)
- CMake 3.18+
- Git



To use GBasis with libcint support, install from source:

```bash
git clone https://github.com/theochem/gbasis.git
cd gbasis
pip install --no-build-isolation -v -e .[dev]
```
During installation, CMake will automatically:

This will automatically:
1. Detect your architecture (x86 with SSE3 → qcint, ARM → libcint)
2. Download and build libcint/qcint v6.1.2 via CMake FetchContent
3. Compile the `libcint_bindings` Python/C extension module

> **Note:** Once released on PyPI, installation will simply be:
> ```bash
> pip install qc-gbasis
> ```

In [20]:
# Verify installation
import gbasis
from gbasis.integrals.libcint import CBasis
print("GBasis imported successfully!")
print("libcint bindings available: True")

GBasis imported successfully!
libcint bindings available: True


## 1. Setup

Import the necessary modules and define the molecule. Atomic coordinates must be in **Bohr** (atomic units).

In [3]:
import numpy as np
import numpy.testing as npt

from gbasis.parsers import make_contractions, parse_nwchem
from gbasis.integrals.libcint import CBasis

In [4]:
# Define a water molecule in Bohr (atomic units)
atsyms = ["O", "H", "H"]
atcoords = np.array([
    [ 0.000000,  0.000000,  0.000000],  # O
    [ 0.000000,  1.430429,  1.107157],  # H
    [ 0.000000, -1.430429,  1.107157],  # H
])

print(f"Molecule: {atsyms}")
print(f"Coordinates (Bohr):\n{atcoords}")

Molecule: ['O', 'H', 'H']
Coordinates (Bohr):
[[ 0.        0.        0.      ]
 [ 0.        1.430429  1.107157]
 [ 0.       -1.430429  1.107157]]


In [5]:
# Parse the STO-3G basis set and build contractions
# Replace 'sto-3g.nwchem' with the path to your basis set file
# GBasis test data: tests/data/data_sto6g.nwchem

#basis_dict = parse_nwchem("/Users/aayushgupta/Desktop/GSOC26/Gbasis/gbasis/tests/data_sto6g.nwchem")
import gbasis
from pathlib import Path

basis_path = Path(gbasis.__file__).parent.parent / "tests" / "data_sto6g.nwchem"
basis_dict = parse_nwchem(str(basis_path))
py_basis = make_contractions(basis_dict, atsyms, atcoords, coord_types="spherical")

# Build the CBasis object (spherical coordinates)
cb = CBasis(py_basis, atsyms, atcoords, coord_type="spherical")

print(f"Number of atoms:           {cb.natm}")
print(f"Number of shells:          {cb.nbas}")
print(f"Number of basis functions: {cb.nbfn}")
print(f"Coordinate type:           {cb.coord_type}")

Number of atoms:           3
Number of shells:          5
Number of basis functions: 7
Coordinate type:           spherical


## 2. One-Electron Integrals

The following 1-electron integrals are available via `CBasis`:

| Method | Integral | Description |
|--------|----------|-------------|
| `cb.overlap()` | $S_{ij} = \langle \phi_i \| \phi_j \rangle$ | Overlap matrix |
| `cb.kinetic_energy()` | $T_{ij} = \langle \phi_i \| -\frac{1}{2}\nabla^2 \| \phi_j \rangle$ | Kinetic energy |
| `cb.nuclear_attraction()` | $V_{ij} = \langle \phi_i \| \sum_A Z_A/r_A \| \phi_j \rangle$ | Nuclear attraction |
| `cb.momentum()` | $p_{ij} = \langle \phi_i \| -i\nabla \| \phi_j \rangle$ | Momentum (complex) |
| `cb.rinv()` | $V_{ij} = \langle \phi_i \| 1/r \| \phi_j \rangle$ | 1/r operator |

In [6]:
# Overlap integral
S = cb.overlap()
print(f"Overlap matrix shape: {S.shape}")
print(f"Overlap diagonal (should be ~1.0 for normalized basis):")
print(np.diag(S).round(6))

# Verify symmetry
npt.assert_allclose(S, S.T, atol=1e-12)
print("Overlap matrix is symmetric: OK")

Overlap matrix shape: (7, 7)
Overlap diagonal (should be ~1.0 for normalized basis):
[1. 1. 1. 1. 1. 1. 1.]
Overlap matrix is symmetric: OK


In [7]:
# Kinetic energy integral
T = cb.kinetic_energy()
print(f"Kinetic energy matrix shape: {T.shape}")
print(f"Kinetic energy diagonal (all positive):")
print(np.diag(T).round(6))
assert np.all(np.diag(T) > 0), "Kinetic energy diagonal should be positive"
print("Kinetic energy diagonal is positive: OK")

Kinetic energy matrix shape: (7, 7)
Kinetic energy diagonal (all positive):
[29.327197  0.842618  2.531309  2.531309  2.531309  0.768522  0.768522]
Kinetic energy diagonal is positive: OK


In [8]:
# Nuclear attraction integral
V = cb.nuclear_attraction()
print(f"Nuclear attraction matrix shape: {V.shape}")
print(f"Nuclear attraction diagonal (all negative):")
print(np.diag(V).round(6))

Nuclear attraction matrix shape: (7, 7)
Nuclear attraction diagonal (all negative):
[-62.363993 -10.091137 -10.007061 -10.164033 -10.1011    -5.848942
  -5.848942]


In [9]:
# Core Hamiltonian H = T + V
H_core = T + V
print(f"Core Hamiltonian diagonal:")
print(np.diag(H_core).round(6))

Core Hamiltonian diagonal:
[-33.036795  -9.248518  -7.475752  -7.632725  -7.569791  -5.08042
  -5.08042 ]


In [10]:
# Momentum integral (purely imaginary: p = -i * real_buffer)
p = cb.momentum(origin=np.zeros(3))
print(f"Momentum integral shape: {p.shape}  (nbfn x nbfn x 3 components)")
print(f"Momentum is complex: {np.iscomplexobj(p)}")

# Momentum is anti-Hermitian: p_ij = -p_ji*
npt.assert_allclose(p[:, :, 0], -p[:, :, 0].conj().T, atol=1e-10)
print("Momentum is anti-Hermitian: OK")

Momentum integral shape: (7, 7, 3)  (nbfn x nbfn x 3 components)
Momentum is complex: True
Momentum is anti-Hermitian: OK


### Optional: MO Transformation

All `CBasis` methods accept an optional `transform` matrix to convert from AO to MO basis.
For example, if `C` is the MO coefficient matrix (shape `[nbfn, nmo]`):

```python
S_mo = cb.overlap(transform=C.T)   # shape: [nmo, nmo]
T_mo = cb.kinetic_energy(transform=C.T)
```

## 3. Electron Repulsion Integrals (ERI)

The 2-electron repulsion integral is:

$$g_{ijkl} = \langle \phi_i \phi_j \| \frac{1}{r_{12}} \| \phi_k \phi_l \rangle$$

Two index conventions are supported:
- **Physicist notation** (default): `out[i, j, k, l]` = $\langle ij | kl \rangle$
- **Chemist notation**: `out[i, j, k, l]` = $(ij|kl)$

In [11]:
# Electron repulsion integrals (physicist notation, default)
eri = cb.electron_repulsion(notation="physicist")
print(f"ERI shape: {eri.shape}  (nbfn x nbfn x nbfn x nbfn)")
print(f"ERI[0,0,0,0] = {eri[0,0,0,0]:.6f}  (should be positive)")
assert np.all(np.isfinite(eri)), "ERI contains NaN or Inf"
print("ERI is finite: OK")

ERI shape: (7, 7, 7, 7)  (nbfn x nbfn x nbfn x nbfn)
ERI[0,0,0,0] = 4.787491  (should be positive)
ERI is finite: OK


In [12]:
# 8-fold permutation symmetry in physicist notation <ij|kl>:
# 1. <ij|kl> = <ji|lk>  (swap i<->j AND k<->l simultaneously)
npt.assert_allclose(eri, eri.transpose(1, 0, 3, 2), atol=1e-10)
# 2. <ij|kl> = <kl|ij>  (swap electron 1 <-> electron 2)
npt.assert_allclose(eri, eri.transpose(2, 3, 0, 1), atol=1e-10)
# 3. <ij|kl> = <lk|ji>  (full reverse)
npt.assert_allclose(eri, eri.transpose(3, 2, 1, 0), atol=1e-10)
# 4. <ij|kl> = <kj|il>  (swap i<->k)
npt.assert_allclose(eri, eri.transpose(2, 1, 0, 3), atol=1e-10)
print("ERI 8-fold symmetry (physicist notation): OK")

ERI 8-fold symmetry (physicist notation): OK


## 4. Moment Integrals

The `moment()` method computes multipole moment integrals up to 3rd order:

$$M_{ij} = \langle \phi_i | (x-X_0)^{n_x}(y-Y_0)^{n_y}(z-Z_0)^{n_z} | \phi_j \rangle$$

In [13]:
origin = np.zeros(3)

# Compute overlap (order 0), dipole (order 1), and quadrupole (order 2) moments
orders = np.array([
    [0, 0, 0],  # overlap
    [1, 0, 0],  # x dipole
    [0, 1, 0],  # y dipole
    [0, 0, 1],  # z dipole
    [2, 0, 0],  # xx quadrupole
    [0, 2, 0],  # yy quadrupole
    [0, 0, 2],  # zz quadrupole
])

M = cb.moment(orders, origin=origin)
print(f"Moment integral shape: {M.shape}  (nbfn x nbfn x n_orders)")

# Order 0 should equal the overlap matrix
npt.assert_allclose(M[:, :, 0], S, atol=1e-10)
print("Moment order 0 == overlap matrix: OK")

Moment integral shape: (7, 7, 7)  (nbfn x nbfn x n_orders)
Moment order 0 == overlap matrix: OK


## 5. Gradient Integrals

Gradient integrals are building blocks for computing nuclear coordinate gradients of the energy.

| Method | libcint function | Description |
|--------|-----------------|-------------|
| `cb.gradient_kinetic()` | `int1e_ipkin` | $i\nabla T$ |
| `cb.gradient_nuclear()` | `int1e_ipnuc` | $i\nabla V$ |
| `cb.gradient_rinv()` | `int1e_iprinv` | $i\nabla (1/r)$ |

In [14]:
# Gradient of kinetic energy integral
ipkin = cb.gradient_kinetic()
print(f"Gradient kinetic shape: {ipkin.shape}")
assert np.all(np.isfinite(ipkin)), "gradient_kinetic contains NaN or Inf"
print(f"gradient_kinetic is finite: OK")

# Gradient of nuclear attraction integral
ipnuc = cb.gradient_nuclear()
print(f"Gradient nuclear shape: {ipnuc.shape}")
assert np.all(np.isfinite(ipnuc))
print(f"gradient_nuclear is finite: OK")

# Gradient of 1/r integral
iprinv = cb.gradient_rinv(inv_origin=np.zeros(3))
print(f"Gradient rinv shape: {iprinv.shape}")
assert np.all(np.isfinite(iprinv))
print(f"gradient_rinv is finite: OK")

Gradient kinetic shape: (7, 7)
gradient_kinetic is finite: OK
Gradient nuclear shape: (7, 7)
gradient_nuclear is finite: OK
Gradient rinv shape: (7, 7)
gradient_rinv is finite: OK


## 6. GIAO / Magnetic Integrals

Gauge-including atomic orbital (GIAO) integrals are building blocks for NMR shielding tensors and magnetic susceptibilities.

| Method | libcint function | Description |
|--------|-----------------|-------------|
| `cb.ia01p()` | `int1e_ia01p` | GIAO paramagnetic shielding |
| `cb.ircxp()` | `int1e_cg_irxp` | GIAO angular momentum |
| `cb.iking()` | `int1e_igkin` | GIAO kinetic energy |
| `cb.iovlpg()` | `int1e_igovlp` | GIAO overlap gradient |
| `cb.inucg()` | `int1e_ignuc` | GIAO nuclear attraction |

In [15]:
# GIAO paramagnetic shielding
ia01p = cb.ia01p()
print(f"ia01p shape: {ia01p.shape}")
assert np.all(np.isfinite(ia01p))
print("ia01p is finite: OK")

# GIAO angular momentum
ircxp = cb.ircxp()
print(f"ircxp shape: {ircxp.shape}")
assert np.all(np.isfinite(ircxp))
print("ircxp is finite: OK")

# GIAO kinetic energy
iking = cb.iking()
print(f"iking shape: {iking.shape}")
assert np.all(np.isfinite(iking))
print("iking is finite: OK")

# GIAO overlap gradient
iovlpg = cb.iovlpg()
print(f"iovlpg shape: {iovlpg.shape}")
assert np.all(np.isfinite(iovlpg))
print("iovlpg is finite: OK")

# GIAO nuclear attraction
inucg = cb.inucg()
print(f"inucg shape: {inucg.shape}")
assert np.all(np.isfinite(inucg))
print("inucg is finite: OK")

ia01p shape: (7, 7)
ia01p is finite: OK
ircxp shape: (7, 7)
ircxp is finite: OK
iking shape: (7, 7)
iking is finite: OK
iovlpg shape: (7, 7)
iovlpg is finite: OK
inucg shape: (7, 7)
inucg is finite: OK


## 7. 3-Center 2-Electron Integrals

The 3-center 2-electron integral is used in density fitting (resolution of the identity) approximations:

$$(ij|k) = \langle \phi_i \phi_j \| \frac{1}{r_{12}} \| \phi_k \rangle$$

This integral exploits $i \leq j$ symmetry: `out[i, j, k] = out[j, i, k]`.

In [16]:
# 3-center 2-electron integrals
int3c2e = cb.three_center_two_electron()
print(f"3c2e shape: {int3c2e.shape}  (nbfn x nbfn x nbfn)")
assert np.all(np.isfinite(int3c2e))
print("3c2e is finite: OK")

# Verify i <-> j symmetry
npt.assert_allclose(int3c2e, int3c2e.transpose(1, 0, 2), atol=1e-10)
print("3c2e i<->j symmetry: OK")

3c2e shape: (7, 7, 7)  (nbfn x nbfn x nbfn)
3c2e is finite: OK
3c2e i<->j symmetry: OK


## 8. Cartesian Basis

`CBasis` also supports cartesian coordinates. Simply pass `coord_type="cartesian"`.

In [17]:
# Build CBasis with cartesian coordinates
py_basis_cart = make_contractions(basis_dict, atsyms, atcoords, coord_types="cartesian")
cb_cart = CBasis(py_basis_cart, atsyms, atcoords, coord_type="cartesian")

print(f"Cartesian basis functions: {cb_cart.nbfn}")
print(f"Spherical basis functions: {cb.nbfn}")

# Overlap diagonal should be 1 for cartesian too
S_cart = cb_cart.overlap()
print(f"Cartesian overlap diagonal:")
print(np.diag(S_cart).round(6))

Cartesian basis functions: 7
Spherical basis functions: 7
Cartesian overlap diagonal:
[1. 1. 1. 1. 1. 1. 1.]


## 9. Verification Against GBasis Python Reference

Here we verify that the C shell-loop results match the pure-Python GBasis reference implementations.

In [18]:
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral_improved
from gbasis.integrals.libcint import ELEMENTS

atnums = np.array([ELEMENTS.index(s) for s in atsyms], dtype=float)
atol = 1e-6

# Overlap
py_S = overlap_integral(py_basis, screen_basis=False)
npt.assert_allclose(cb.overlap(), py_S, atol=atol)
print("Overlap matches Python reference: OK")

# Kinetic energy
py_T = kinetic_energy_integral(py_basis, screen_basis=False)
npt.assert_allclose(cb.kinetic_energy(), py_T, atol=atol)
print("Kinetic energy matches Python reference: OK")

# Nuclear attraction
py_V = nuclear_electron_attraction_integral(py_basis, atcoords, atnums)
npt.assert_allclose(cb.nuclear_attraction(), py_V, atol=atol)
print("Nuclear attraction matches Python reference: OK")

# ERI (looser tolerance — 4-index integral)
py_eri = electron_repulsion_integral_improved(py_basis)
npt.assert_allclose(cb.electron_repulsion(), py_eri, atol=1e-4, rtol=1e-5)
print("ERI matches Python reference: OK")

Overlap matches Python reference: OK
Kinetic energy matches Python reference: OK
Nuclear attraction matches Python reference: OK
ERI matches Python reference: OK


## 10. Benchmarks

All benchmarks run on **MacBook Air M5** (Apple Silicon, ARM), H₂O molecule,
spherical coordinates, averaged over 50 iterations.
Run the code cell below to reproduce on your machine.

> **Note:** Results are pre-run; actual values may vary by hardware.

### STO-6G Basis (7 basis functions)

| Integral | Python loops | C shell-loop | Speedup |
|----------|-------------|--------------|---------|
| Overlap  | 3.126 ms | 0.009 ms | **340x** |
| Kinetic  | 3.225 ms | 0.019 ms | **166x** |
| Nuclear  | 3.604 ms | 0.025 ms | **147x** |

### cc-pVDZ Basis (24 basis functions)

| Integral | Python loops | C shell-loop | Speedup |
|----------|-------------|--------------|---------|
| Overlap  | 6.802 ms | 0.019 ms | **353x** |
| Kinetic  | 7.486 ms | 0.036 ms | **211x** |
| Nuclear  | 9.075 ms | 0.047 ms | **192x** |

The C shell-loop implementation achieves **147x–353x speedup** over the
pure-Python baseline for 1-electron integrals. Speedup increases with
basis set size as Python loop overhead grows while the C implementation
scales efficiently.

In [19]:
import numpy as np
import time
import os
from gbasis.parsers import make_contractions, parse_nwchem
from gbasis.integrals.libcint import CBasis
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral

# Basis set paths
TEST_DIR = "/Users/aayushgupta/Desktop/GSOC26/Gbasis/gbasis/tests"
STO6G   = os.path.join(TEST_DIR, "data_sto6g.nwchem")
CCPVDZ  = os.path.join(TEST_DIR, "data_ccpvdz.nwchem")

atsyms   = ["O", "H", "H"]
atcoords = np.array([
    [0.0,  0.000000,  0.000000],
    [0.0,  1.430429,  1.107157],
    [0.0, -1.430429,  1.107157],
])
atnums = np.array([8.0, 1.0, 1.0])

N = 50  # iterations per benchmark

print(f"Molecule: H2O | Iterations: {N} | Hardware: MacBook Air M5 (ARM)\n")

for basis_file, basis_name in [(STO6G, "STO-6G"), (CCPVDZ, "cc-pVDZ")]:
    basis_dict = parse_nwchem(basis_file)
    py_basis   = make_contractions(basis_dict, atsyms, atcoords, coord_types="spherical")
    cb         = CBasis(py_basis, atsyms, atcoords, coord_type="spherical")

    print(f"--- {basis_name} ({cb.nbfn} basis functions) ---")
    for name, py_fn, c_fn in [
        ("overlap", lambda: overlap_integral(py_basis, screen_basis=False), cb.overlap),
        ("kinetic", lambda: kinetic_energy_integral(py_basis, screen_basis=False), cb.kinetic_energy),
        ("nuclear", lambda: nuclear_electron_attraction_integral(py_basis, atcoords, atnums), cb.nuclear_attraction),
    ]:
        t0 = time.perf_counter()
        for _ in range(N): py_fn()
        py_t = (time.perf_counter() - t0) / N

        t0 = time.perf_counter()
        for _ in range(N): c_fn()
        c_t = (time.perf_counter() - t0) / N

        print(f"  {name:8s}: Python={py_t*1000:.3f}ms  C={c_t*1000:.3f}ms  Speedup={py_t/c_t:.0f}x")
    print()

Molecule: H2O | Iterations: 50 | Hardware: MacBook Air M5 (ARM)

--- STO-6G (7 basis functions) ---
  overlap : Python=2.942ms  C=0.009ms  Speedup=327x
  kinetic : Python=3.177ms  C=0.020ms  Speedup=162x
  nuclear : Python=3.559ms  C=0.024ms  Speedup=148x

--- cc-pVDZ (24 basis functions) ---
  overlap : Python=6.529ms  C=0.019ms  Speedup=339x
  kinetic : Python=7.164ms  C=0.035ms  Speedup=203x
  nuclear : Python=8.385ms  C=0.042ms  Speedup=199x



## Summary

| Integral | Method | Shape | Notes |
|----------|--------|-------|-------|
| Overlap | `cb.overlap()` | `(N, N)` | Symmetric |
| Kinetic energy | `cb.kinetic_energy()` | `(N, N)` | Symmetric, positive diagonal |
| Nuclear attraction | `cb.nuclear_attraction()` | `(N, N)` | Symmetric |
| 1/r | `cb.rinv()` | `(N, N)` | Symmetric |
| Momentum | `cb.momentum()` | `(N, N, 3)` | Complex, anti-Hermitian |
| Moment | `cb.moment(orders)` | `(N, N, M)` | M = number of orders |
| ERI | `cb.electron_repulsion()` | `(N, N, N, N)` | 8-fold symmetry |
| Gradient kinetic | `cb.gradient_kinetic()` | `(N, N)` | |
| Gradient nuclear | `cb.gradient_nuclear()` | `(N, N)` | |
| Gradient rinv | `cb.gradient_rinv()` | `(N, N)` | |
| GIAO ia01p | `cb.ia01p()` | `(N, N)` | NMR building block |
| GIAO ircxp | `cb.ircxp()` | `(N, N)` | NMR building block |
| GIAO iking | `cb.iking()` | `(N, N)` | NMR building block |
| GIAO iovlpg | `cb.iovlpg()` | `(N, N)` | NMR building block |
| GIAO inucg | `cb.inucg()` | `(N, N)` | NMR building block |
| 3c2e | `cb.three_center_two_electron()` | `(N, N, N)` | i<=j symmetry |

All methods accept an optional `transform` matrix to convert from AO to MO basis.